# Trabalho M2 — Processamento de Imagens
## Pipeline de Segmentação de Grãos Agrícolas

**Dataset:** Seed Images (Kaggle — `ddsssss/seed-images`) — imagens de grãos sobre placa de Petri
**Disciplina:** Processamento de Imagens — UNIVALI
**Professor:** Felipe Viel
**Autores:** Eduardo Drux e Neto

---

## Enunciado

Desenvolver um pipeline completo de processamento digital de imagens para segmentar
automaticamente os grãos de interesse em cada imagem, gerando uma máscara binária onde
**pixels de grão = 1** e **pixels de fundo = 0**. Avaliar com IoU, Dice e contagem de grãos.

## Contexto

O dataset escolhido é composto por imagens (1920×1080) de grãos (sementes) sobre uma
placa de Petri sobre fundo colorido. Os principais desafios:

- Os grãos **tocam uns aos outros** com frequência, formando aglomerados.
- A borda da placa e a vinheta da lente fisheye geram falsos positivos.
- Há **reflexos e brilhos** no vidro da placa.
- Variação de iluminação entre imagens.
- Anotações são **bounding boxes** (PASCAL VOC), não máscaras pixel-a-pixel — então a
  avaliação usa GT aproximado por elipses inscritas nas caixas.

## Fluxo escolhido

```
Imagem RGB
  ↓  1. Pré-processamento     → RGB → LAB, normalização
  ↓  2. Filtragem na frequência → Passa-baixa Gaussiano via FFT no canal b*
  ↓  3. Construção da feature → b* "iluminado" (b* * gate(L*)) — anula vinheta
  ↓  4. SLIC Superpixels       → from scratch, no espaço LAB
  ↓  5. Otsu por Superpixel    → from scratch, sobre médias dos superpixels
  ↓  6. Morfologia matemática  → from scratch: abertura, filtro de área, fill holes, fechamento
  ↓  7. Contagem               → from scratch: erosão + componentes conectados (BFS)
  ↓  8. Avaliação              → IoU, Dice (from scratch) + contagem
```

## Por que esse fluxo

- **LAB**: o canal b* (eixo azul–amarelo) é praticamente uma separação linear entre
  grãos (amarelados, b* alto) e fundo teal/verde (b* baixo). Tentei a* e a luminância
  isoladamente — o b* sozinho dá a melhor relação sinal/ruído.
- **Passa-baixa Gaussiano**: a textura interna dos grãos é alta frequência e não
  ajuda a segmentação — suavizar melhora a estabilidade do SLIC e do Otsu.
- **SLIC**: gera regiões compactas e regulares; comparado a Watershed (que explode em
  bacias) ou Felzenszwalb (que é instável com textura), o SLIC é o algoritmo mais
  previsível pra esse tipo de imagem.
- **Otsu por superpixel**: a média intra-superpixel já filtra ruído, então o histograma
  de médias é muito mais bimodal que o histograma de pixels — Otsu acerta o limiar
  com mais facilidade.
- **Morfologia from scratch**: o enunciado exige. Uso *abertura* pra ruído pontual,
  *filtro por área* pra eliminar a borda da placa, *fill holes* pra fechar buracos
  internos e *fechamento* pra suavizar bordas.


## Bloco 1 — Importações e configuração

### Bibliotecas usadas (e por quê cada uma é permitida)

| Biblioteca | Uso | Justificativa |
|---|---|---|
| `numpy` | Operações matriciais | Base obrigatória — sem ela nada anda |
| `matplotlib` | Visualização | Permitido (não é segmentação) |
| `PIL` (Pillow) | Carregar JPG e redimensionar | Permitido (I/O) |
| `xml.etree` | Ler XMLs do dataset | Permitido (parsing) |
| `collections.deque` | Fila eficiente para BFS | Permitido (estrutura de dados) |
| `numpy.fft` | FFT no domínio da frequência | **Permitido explicitamente** pelo enunciado |
| `skimage.color.rgb2lab` | Conversão RGB → LAB | É pré-processamento de espaço de cor (não é segmentação nem morfologia). Implementar a fórmula CIE LAB do zero não agregaria valor ao trabalho. |

### Por que redimensionar pra 640×360

O original é 1920×1080 (~2 milhões de pixels). O SLIC e as operações morfológicas
em Python puro processam pixel a pixel — em 1920×1080 cada iteração levaria minutos.
Redimensionando pra 640×360 reduzo em ~9× o volume de dados, mantendo grãos com
dezenas de pixels (largura ainda ~12-18 px), suficiente pra todas as etapas.


In [ ]:
import os
import random
import xml.etree.ElementTree as ET
from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage.color import rgb2lab

# ── Configuração portátil do projeto ──────────────────────────────────────────
# Este notebook deve ficar na raiz do repositório.
# Assim, Path.cwd() aponta para a pasta do projeto em qualquer máquina.
BASE_PATH = Path.cwd()

DATASET_PATH = BASE_PATH / "dataset"
IMAGES_PATH = DATASET_PATH / "JPEGImages"
ANNOTATIONS_PATH = DATASET_PATH / "Annotations"
FIG_PATH = BASE_PATH / "figures"

# Cria a pasta de figuras caso ela ainda não exista.
FIG_PATH.mkdir(exist_ok=True)

# ── Imagens que vou usar nos experimentos principais ─────────────────────────
# Escolhi essas porque cobrem situações diferentes do dataset:
#   0619 — placa centralizada, fundo teal limpo (caso "fácil")
#   1105 — vinheta verde clara, poucos grãos esparsos
#   1113 — vinheta escura forte (caso "difícil" para Otsu no b* puro)
#   1141 — placa centralizada com vinheta verde
#   1286 — muitos grãos colados em cluster (caso "difícil" para contagem)
SELECTED_IMAGES = ["0619", "1105", "1113", "1141", "1286"]

# Tamanho de trabalho — ~9x menos pixels que o original.
RESIZE_TO = (640, 360)

print("Bibliotecas carregadas.")
print("BASE_PATH:", BASE_PATH)
print("IMAGES_PATH:", IMAGES_PATH)
print("ANNOTATIONS_PATH:", ANNOTATIONS_PATH)
print("FIG_PATH:", FIG_PATH)
print(f"Imagens selecionadas: {SELECTED_IMAGES}")
print(f"Tamanho de processamento: {RESIZE_TO[0]}x{RESIZE_TO[1]} px")

# Verificação rápida para evitar erro silencioso caso o notebook seja aberto
# em uma pasta errada ou o dataset não esteja no lugar esperado.
assert IMAGES_PATH.exists(), f"Pasta de imagens não encontrada: {IMAGES_PATH}"
assert ANNOTATIONS_PATH.exists(), f"Pasta de anotações não encontrada: {ANNOTATIONS_PATH}"

jpg_files = sorted(IMAGES_PATH.glob("*.jpg"))
xml_files = sorted(ANNOTATIONS_PATH.glob("*.xml"))

print(f"Total de imagens JPG: {len(jpg_files)}")
print(f"Total de anotações XML: {len(xml_files)}")

assert len(jpg_files) > 0, "Nenhuma imagem JPG encontrada."
assert len(xml_files) > 0, "Nenhum XML encontrado."
assert len(jpg_files) == len(xml_files), "Quantidade de JPG e XML está diferente."

missing_xml = [p.stem for p in jpg_files if not (ANNOTATIONS_PATH / f"{p.stem}.xml").exists()]
missing_jpg = [p.stem for p in xml_files if not (IMAGES_PATH / f"{p.stem}.jpg").exists()]

assert not missing_xml, f"Há imagens sem XML correspondente. Exemplos: {missing_xml[:10]}"
assert not missing_jpg, f"Há XMLs sem JPG correspondente. Exemplos: {missing_jpg[:10]}"

print("Estrutura do dataset OK.")


## Bloco 2 — Carregamento de imagens e ground truth

O dataset traz as anotações no formato **PASCAL VOC** (XML), com bounding boxes para
cada grão. Não há máscara pixel-a-pixel.

Pra conseguir calcular IoU e Dice mesmo sem máscara real, eu monto um **GT
aproximado**: cada bndbox vira uma elipse inscrita. Por que elipse e não retângulo?
Os grãos são alongados/oval, então a elipse aproxima muito melhor a forma real do
que um retângulo preenchido (que ficaria com cantos cheios de fundo).

Isso não é GT perfeito — é a melhor aproximação possível com o que o dataset oferece.
Vou discutir as implicações disso na seção de análise.


In [ ]:
def load_image(image_id, resize_to=RESIZE_TO):
    """Lê um JPG do dataset, garante RGB e redimensiona."""
    path = os.path.join(IMAGES_PATH, f"{image_id}.jpg")
    img = Image.open(path).convert("RGB")
    img = img.resize(resize_to, Image.BILINEAR)
    return np.array(img, dtype=np.uint8)


def parse_annotation(image_id, target_w, target_h):
    """Lê o XML PASCAL VOC e escala as bndbox pro tamanho redimensionado.

    Retorna lista de (xmin, ymin, xmax, ymax, label).
    """
    path = os.path.join(ANNOTATIONS_PATH, f"{image_id}.xml")
    root = ET.parse(path).getroot()

    orig_w = int(root.find("size/width").text)
    orig_h = int(root.find("size/height").text)
    sx = target_w / orig_w
    sy = target_h / orig_h

    boxes = []
    for obj in root.findall("object"):
        bb = obj.find("bndbox")
        xmin = int(int(bb.find("xmin").text) * sx)
        ymin = int(int(bb.find("ymin").text) * sy)
        xmax = int(int(bb.find("xmax").text) * sx)
        ymax = int(int(bb.find("ymax").text) * sy)
        label = obj.find("name").text  # "yes"/"no" = germinou ou não
        boxes.append((xmin, ymin, xmax, ymax, label))
    return boxes


def build_gt_mask(boxes, shape):
    """Constrói máscara binária de GT a partir das bndbox usando elipses inscritas.

    Cada caixa vira a maior elipse que cabe dentro dela. Faço isso de forma
    vetorizada — calculo a equação da elipse num grid de coordenadas e somo
    (OR) na máscara final.
    """
    H, W = shape
    mask = np.zeros((H, W), dtype=np.uint8)
    y_idx = np.arange(H).reshape(-1, 1)
    x_idx = np.arange(W).reshape(1, -1)

    for (xmin, ymin, xmax, ymax, _label) in boxes:
        cx = (xmin + xmax) / 2.0
        cy = (ymin + ymax) / 2.0
        rx = max(1.0, (xmax - xmin) / 2.0)
        ry = max(1.0, (ymax - ymin) / 2.0)
        ellipse = ((x_idx - cx) / rx) ** 2 + ((y_idx - cy) / ry) ** 2 <= 1.0
        mask |= ellipse.astype(np.uint8)
    return mask


# ── Demonstração: carrega a primeira imagem e visualiza ───────────────────────
img_demo = load_image("0619")
boxes_demo = parse_annotation("0619", img_demo.shape[1], img_demo.shape[0])
gt_demo = build_gt_mask(boxes_demo, img_demo.shape[:2])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img_demo);             axes[0].set_title("Imagem 0619")
axes[1].imshow(img_demo);             axes[1].set_title(f"Imagem + {len(boxes_demo)} bbox")
# desenha caixas sobre a imagem
for (xmin, ymin, xmax, ymax, _l) in boxes_demo:
    axes[1].add_patch(plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                                    fill=False, edgecolor="yellow", lw=1))
axes[2].imshow(gt_demo, cmap="gray"); axes[2].set_title("GT (elipses inscritas)")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "01_carregamento.png"))
plt.show()
print(f"Imagem {img_demo.shape}, {len(boxes_demo)} grãos anotados, GT cobre {gt_demo.sum()} px")


## Bloco 3 — Pré-processamento e filtragem na frequência

### 3.1 Conversão para LAB

LAB é um espaço de cor **perceptualmente uniforme**, onde:
- `L*` = luminosidade (0–100)
- `a*` = eixo verde–vermelho
- `b*` = eixo **azul–amarelo** ← é o canal que mais separa grão de fundo nessas imagens

Os grãos são amarelados (b* alto), o fundo da placa é teal/azulado (b* baixo ou
negativo). Então **b* sozinho** já é um ótimo discriminante. Olhei também a* mas
ele varia muito menos.

### 3.2 Filtragem passa-baixa Gaussiana via FFT (etapa de "frequência")

Aplico um filtro passa-baixa Gaussiano no canal b*. Por quê:

- **Por que passa-baixa**: a textura interna dos grãos e o ruído do sensor são
  alta frequência. Suavizando antes do SLIC, cada grão fica com b* mais
  homogêneo — os superpixels saem mais "limpos".
- **Por que Gaussiano e não ideal**: filtro passa-baixa ideal (corte abrupto)
  gera **ringing** (anéis) por causa do fenômeno de Gibbs. Gaussiana é suave e
  não introduz artefato.
- **Por que via FFT e não convolução espacial**: a Gaussiana espacial precisa de
  um kernel grande (raio ~3σ ≈ 75 pixels com σ=25). Convolução com kernel 151×151
  custa caro; FFT resolve isso em O(N log N) independente do σ.
- **σ = 25**: testei 10, 15, 25, 40 — abaixo de 15 a suavização é fraca e o
  Otsu ainda pega ruído; acima de 40 a borda do grão fica indefinida.

### 3.3 Feature "warmth gated by L*"

Algumas imagens (1113, 1141) têm **vinheta escura** nas bordas — o b* nessas
regiões fica próximo de zero, o que confunde o Otsu (que pode classificá-las
como grão).

Resolvo combinando b* com a luminosidade L*: defino um **gate** linear que
vai de 0 (quando L<20) até 1 (quando L>40). Multiplicado em b*, isso zera a
contribuição da vinheta sem afetar o resto da imagem.

```
feature(x,y) = b*(x,y) × clip((L*(x,y) - 20) / 20, 0, 1)
```


In [ ]:
def to_lab(img_rgb):
    """RGB uint8 [0,255] → LAB float32. Permitido (não é segmentação)."""
    return rgb2lab(img_rgb.astype(np.float32) / 255.0).astype(np.float32)


def gaussian_lowpass_fft(channel, sigma=25.0):
    """Filtro passa-baixa Gaussiano via FFT (uso de biblioteca permitido na frequência).

    Passos:
      1. FFT 2D do canal, com fftshift pra DC ficar no centro (mais intuitivo).
      2. Monta um Gaussiano centrado em (H/2, W/2) no plano da frequência.
      3. Multiplica o espectro pelo Gaussiano (= convolução no espacial).
      4. Inverte (ifftshift + ifft2) e pega a parte real.
    """
    H, W = channel.shape
    F = np.fft.fftshift(np.fft.fft2(channel))

    cy, cx = H // 2, W // 2
    y, x = np.indices((H, W))
    D2 = (y - cy) ** 2 + (x - cx) ** 2
    H_filter = np.exp(-D2 / (2.0 * sigma * sigma))

    G = F * H_filter
    out = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
    return out.astype(np.float32)


def warmth_feature(img_lab, l_low=20.0, l_high=40.0):
    """Feature de classificação = b* com gate suave em L*.

    Anula o sinal nas regiões muito escuras (vinheta) e deixa o resto passar.
    """
    L = img_lab[..., 0]
    b = img_lab[..., 2]
    gate = np.clip((L - l_low) / max(l_high - l_low, 1e-6), 0.0, 1.0)
    return b * gate


# ── Demonstração: comparar b* puro vs b* suavizado vs feature warmth ──────────
img = load_image("1113")  # imagem com vinheta escura forte — caso difícil
lab = to_lab(img)
b_raw = lab[..., 2]
b_smooth = gaussian_lowpass_fft(b_raw, sigma=25.0)
lab_smooth = lab.copy()
lab_smooth[..., 2] = b_smooth
feat = warmth_feature(lab_smooth)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].imshow(img);                            axes[0].set_title("RGB original (1113)")
axes[1].imshow(b_raw, cmap="RdYlBu_r");         axes[1].set_title("b* puro")
axes[2].imshow(b_smooth, cmap="RdYlBu_r");      axes[2].set_title("b* após passa-baixa σ=25")
axes[3].imshow(feat, cmap="RdYlBu_r");          axes[3].set_title("feature = b* × gate(L*)")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "02_frequencia_e_feature.png"))
plt.show()

print(f"b* puro:     min={b_raw.min():.1f}, max={b_raw.max():.1f}")
print(f"b* suavizado: min={b_smooth.min():.1f}, max={b_smooth.max():.1f}")
print(f"feature:     min={feat.min():.1f}, max={feat.max():.1f}")


## Bloco 4 — SLIC Superpixels (from scratch)

**Referência:** Achanta et al., *SLIC Superpixels Compared to State-of-the-Art Superpixel
Methods*, TPAMI 2012.

### Por que SLIC e não outro algoritmo

| Algoritmo | Problema pra este dataset |
|---|---|
| Watershed sem marcadores | Explode em milhares de bacias por causa de ruído |
| Felzenszwalb | Regiões muito irregulares; instável com textura do fundo |
| Quickshift | Lento (O(dN²)) e parâmetros muito sensíveis |
| **SLIC** | Compacto, regular, O(N), parâmetros intuitivos ✓ |

### Como o SLIC funciona

1. **Inicialização**: distribuo K centros numa **grade regular** sobre a imagem.
   Cada centro guarda um vetor 5D: `[L, a, b, x, y]`.

2. **Espaçamento**: `S = sqrt(N/K)`. Cada centro só "olha" pixels dentro de uma
   janela `2S × 2S` ao redor dele. Isso é o que torna o SLIC O(N) em vez de
   O(N·K) como o k-means tradicional.

3. **Atribuição**: pra cada pixel da janela, calculo a distância híbrida ao
   centro:

   ```
   D² = (ΔL² + Δa² + Δb²) + (m/S)² × (Δx² + Δy²)
   ```

   onde `m` é a **compactness**. Maior m → superpixels mais quadrados/regulares.
   Menor m → fronteiras seguem mais a cor (mais "irregulares").

4. **Atualização**: cada centro vira a média dos pixels que foram atribuídos a ele.

5. **Repete** 3-4 por algumas iterações (8-10 já estabiliza).

### Parâmetros escolhidos

| Parâmetro | Valor | Por quê |
|---|---|---|
| `n_segments` | 300 | Suficiente pra grãos ficarem cobertos por 1-3 superpixels |
| `compactness` | 12 | Bom equilíbrio — não tão duro que ignora cor, não tão solto que vira blob |
| `n_iters` | 8 | Empiricamente os centros estabilizam em 6-8 iterações |

### Por que LAB e não RGB no SLIC

A distância euclidiana em LAB é **perceptualmente uniforme** (ΔE ≈ percepção de
diferença). Em RGB, diferenças iguais em valor numérico não correspondem a
diferenças iguais perceptuais — o SLIC fica enviesado pelos canais G e B.


In [ ]:
def slic_superpixels(img_lab, n_segments=300, compactness=12.0, n_iters=8):
    """SLIC implementado do zero — só numpy.

    Devolve um array (H, W) int32 onde cada pixel tem o id do seu superpixel.
    """
    H, W, _ = img_lab.shape
    N = H * W
    S = max(2, int(np.sqrt(N / n_segments)))  # espaçamento da grade

    # 1) Inicializa centros — grade regular cobrindo a imagem
    centers = []
    for cy in range(S // 2, H, S):
        for cx in range(S // 2, W, S):
            l, a, b = img_lab[cy, cx]
            centers.append([float(l), float(a), float(b), float(cx), float(cy)])
    centers = np.array(centers, dtype=np.float32)
    K = len(centers)

    labels = -np.ones((H, W), dtype=np.int32)
    distances = np.full((H, W), np.inf, dtype=np.float32)

    # Coords dos pixels — pré-calculadas pra não recriar em cada iteração
    ys_full, xs_full = np.indices((H, W), dtype=np.int32)
    m_over_S = compactness / float(S)

    for _it in range(n_iters):
        labels.fill(-1)
        distances.fill(np.inf)

        # 2-3) Atribui cada pixel da janela 2S×2S ao centro mais próximo
        for k in range(K):
            cl, ca, cb, cx, cy = centers[k]
            icx, icy = int(cx), int(cy)
            y0 = max(0, icy - S); y1 = min(H, icy + S + 1)
            x0 = max(0, icx - S); x1 = min(W, icx + S + 1)

            window = img_lab[y0:y1, x0:x1]
            dl = window[..., 0] - cl
            da = window[..., 1] - ca
            db = window[..., 2] - cb
            d_color = dl * dl + da * da + db * db

            ys = ys_full[y0:y1, x0:x1] - cy
            xs = xs_full[y0:y1, x0:x1] - cx
            d_space = xs * xs + ys * ys

            # D ao quadrado — não preciso da raiz só pra comparar
            D = d_color + (m_over_S ** 2) * d_space

            local_d = distances[y0:y1, x0:x1]
            update = D < local_d
            local_d[update] = D[update]
            labels[y0:y1, x0:x1] = np.where(update, k, labels[y0:y1, x0:x1])

        # 4) Atualiza cada centro = média dos pixels atribuídos
        flat = labels.ravel()
        counts = np.bincount(flat, minlength=K).astype(np.float32)
        counts_safe = np.where(counts == 0, 1.0, counts)

        new_centers = np.empty_like(centers)
        for ch in range(3):
            new_centers[:, ch] = np.bincount(flat,
                weights=img_lab[..., ch].ravel(), minlength=K) / counts_safe
        new_centers[:, 3] = np.bincount(flat,
            weights=xs_full.ravel().astype(np.float32), minlength=K) / counts_safe
        new_centers[:, 4] = np.bincount(flat,
            weights=ys_full.ravel().astype(np.float32), minlength=K) / counts_safe
        centers = new_centers

    # Garante que nenhum pixel ficou sem label (pode acontecer se a janela do
    # centro não cobrir algum pixel — raríssimo com S correto)
    if (labels < 0).any():
        for y in range(H):
            for x in range(W):
                if labels[y, x] < 0:
                    labels[y, x] = labels[y-1, x] if y > 0 else (labels[y, x-1] if x > 0 else 0)
    return labels


# ── Demonstração: SLIC numa das imagens ──────────────────────────────────────
img = load_image("0619")
lab = to_lab(img)
lab[..., 2] = gaussian_lowpass_fft(lab[..., 2], sigma=25.0)

import time
t0 = time.time()
sp = slic_superpixels(lab, n_segments=300, compactness=12.0, n_iters=8)
print(f"SLIC rodou em {time.time()-t0:.2f}s — {sp.max()+1} superpixels gerados")

# Sobreposição das fronteiras dos superpixels sobre a imagem original
# Borda de superpixel = pixel com vizinho de label diferente
edges = np.zeros(sp.shape, dtype=bool)
edges[:-1, :] |= sp[:-1, :] != sp[1:, :]
edges[:, :-1] |= sp[:, :-1] != sp[:, 1:]
overlay = img.copy()
overlay[edges] = [255, 0, 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img);                       axes[0].set_title("Original")
axes[1].imshow(sp % 47, cmap="tab20");     axes[1].set_title(f"Labels SLIC ({sp.max()+1} sp)")
axes[2].imshow(overlay);                   axes[2].set_title("Fronteiras sobre original")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "03_slic.png"))
plt.show()


## Bloco 5 — Otsu por Superpixel (from scratch)

**Referência:** Otsu, *A Threshold Selection Method from Gray-Level Histograms*,
IEEE TSMC 1979.

### A ideia

Em vez de aplicar Otsu pixel-a-pixel, **agrupo pixels por superpixel, calculo a
média do canal feature dentro de cada superpixel, e aplico Otsu no vetor de médias**.

### Por que isso funciona melhor que Otsu pixel-a-pixel

- A média intra-superpixel já filtra muito do ruído pixel-a-pixel.
- O histograma de médias é **muito mais bimodal** que o histograma de pixels —
  Otsu acha o limiar correto com mais confiança.
- O resultado é uma classificação **por superpixel** (cada superpixel é grão ou
  fundo inteiro), o que já dá uma máscara com fronteiras mais coerentes.

### Otsu em uma frase

Otsu escolhe o limiar `t` que **maximiza a variância entre as duas classes**
resultantes (= **minimiza a variância dentro de cada classe**). Matematicamente:

```
σ²_b(t) = w₀(t) · w₁(t) · (μ₀(t) − μ₁(t))²
```

onde `w` é a proporção de cada classe e `μ` é a média de cada classe. Testo
todos os bins do histograma e escolho o `t` com maior σ²_b.

### Implementação eficiente

Em vez de recalcular w0, μ0 para cada t (O(N²)), uso somas acumuladas (O(N)):

- `w₀(t) = cumsum(p)[t]`
- `μ₀(t) = cumsum(p·centro)[t] / w₀(t)`
- `μ₁(t) = (μ_total − cumsum(p·centro)[t]) / (1 − w₀(t))`


In [ ]:
def superpixel_mean(labels, channel):
    """Média do canal dentro de cada superpixel — vetorizado com bincount."""
    K = int(labels.max()) + 1
    flat = labels.ravel()
    weights = channel.ravel().astype(np.float64)
    sums = np.bincount(flat, weights=weights, minlength=K)
    counts = np.bincount(flat, minlength=K).astype(np.float64)
    counts = np.where(counts == 0, 1, counts)
    return sums / counts


def otsu_threshold(values, n_bins=256):
    """Otsu manual: maximiza variância inter-classes em 1D.

    Implementação vetorizada com prefix sums — O(n_bins) em vez de O(n_bins²).
    """
    vmin = float(values.min())
    vmax = float(values.max())
    if vmax - vmin < 1e-8:
        return vmin

    hist, edges = np.histogram(values, bins=n_bins, range=(vmin, vmax))
    centers = (edges[:-1] + edges[1:]) / 2.0
    p = hist.astype(np.float64) / hist.sum()

    # Prefix sums — calculo w0 e mu0 acumulados de uma vez
    w0 = np.cumsum(p)
    mu_acc = np.cumsum(p * centers)
    mu_total = mu_acc[-1]
    w1 = 1.0 - w0
    valid = (w0 > 1e-8) & (w1 > 1e-8)

    mu0 = np.where(valid, mu_acc / np.where(w0 == 0, 1, w0), 0.0)
    mu1 = np.where(valid, (mu_total - mu_acc) / np.where(w1 == 0, 1, w1), 0.0)
    sigma_b2 = np.where(valid, w0 * w1 * (mu0 - mu1) ** 2, -1.0)

    best_idx = int(np.argmax(sigma_b2))
    return float(centers[best_idx])


def otsu_per_superpixel(labels, feature):
    """Classifica cada superpixel via Otsu sobre a média da feature.

    Devolve máscara binária no tamanho da imagem.
    """
    sp_mean = superpixel_mean(labels, feature)
    t = otsu_threshold(sp_mean)
    sp_class = (sp_mean > t).astype(np.uint8)
    return sp_class[labels], t, sp_mean


# ── Demonstração ─────────────────────────────────────────────────────────────
img = load_image("0619")
lab = to_lab(img)
lab[..., 2] = gaussian_lowpass_fft(lab[..., 2], sigma=25.0)
feat = warmth_feature(lab)
sp = slic_superpixels(lab, n_segments=300, compactness=12.0, n_iters=8)
coarse, t_otsu, sp_means = otsu_per_superpixel(sp, feat)

print(f"Otsu escolheu t = {t_otsu:.2f} (sobre {len(sp_means)} superpixels)")
print(f"Superpixels classificados como grão: {(sp_means > t_otsu).sum()} / {len(sp_means)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img); axes[0].set_title("Original")
axes[0].axis("off")
axes[1].hist(sp_means, bins=40, color="steelblue", edgecolor="black")
axes[1].axvline(t_otsu, color="red", linestyle="--", label=f"Otsu t={t_otsu:.1f}")
axes[1].set_title("Histograma das médias por superpixel")
axes[1].set_xlabel("feature média"); axes[1].set_ylabel("contagem")
axes[1].legend()
axes[2].imshow(coarse, cmap="gray"); axes[2].set_title("Máscara coarse (Otsu)")
axes[2].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "04_otsu_superpixel.png"))
plt.show()


## Bloco 6 — Morfologia matemática (from scratch)

Implementei do zero todas as operações morfológicas necessárias:

| Operação | Pra que serve aqui |
|---|---|
| Erosão | Encolhe a máscara — uso na contagem pra separar grãos colados |
| Dilatação | Expande a máscara — usado como passo intermediário |
| Abertura (eros+dilat) | Remove pingos isolados (ruído pontual) sem mudar o tamanho dos grãos |
| Fechamento (dilat+eros) | Suaviza bordas e cola "pontes" finas |
| Fill holes | Preenche buracos internos dos grãos (sombras, reflexos) |
| Filtro por área | Remove componentes muito pequenos (ruído) ou muito grandes (borda da placa) |

### Por que disco e não quadrado como SE

Os grãos não têm orientação preferencial — usar um SE quadrado deixa "cantos"
visíveis na máscara e enviesa a forma final. O disco trata todas as direções igual.

### Implementação vetorizada

Em vez de loop pixel-a-pixel (que seria proibitivo em Python), uso o seguinte truque:

- **Erosão** = pra cada offset `(dx, dy)` ativo no SE, faço AND lógico da máscara
  deslocada com o resultado parcial. Pra um SE com K pontos, são K operações
  vetorizadas em vez de H·W·K operações escalares.

- **Dilatação** = mesma coisa, mas OR no lugar de AND.

### Fill holes — truque clássico

1. Faço **flood fill** (BFS) a partir do canto da imagem.
2. Tudo que **não foi alcançado** e era fundo na máscara original é um buraco
   fechado dentro de algum objeto.
3. Inverto isso e adiciono à máscara.

Por que BFS iterativo: recursão estouraria stack em imagens grandes (640×360 = 230k pixels).


In [ ]:
def disk_se(radius):
    """Elemento estruturante em forma de disco (raio inteiro)."""
    r = int(radius)
    y, x = np.ogrid[-r:r+1, -r:r+1]
    return (x*x + y*y <= r*r).astype(np.uint8)


def erode(mask, se):
    """Erosão binária — pixel só fica 1 se TODOS os vizinhos definidos pelo SE forem 1."""
    H, W = mask.shape
    sH, sW = se.shape
    pad_h, pad_w = sH // 2, sW // 2
    padded = np.pad(mask, ((pad_h, pad_h), (pad_w, pad_w)),
                    mode="constant", constant_values=0)
    out = np.ones((H, W), dtype=np.uint8)
    for dy in range(sH):
        for dx in range(sW):
            if se[dy, dx]:
                out &= padded[dy:dy+H, dx:dx+W]
    return out


def dilate(mask, se):
    """Dilatação binária — pixel vira 1 se ALGUM vizinho definido pelo SE for 1."""
    H, W = mask.shape
    sH, sW = se.shape
    pad_h, pad_w = sH // 2, sW // 2
    padded = np.pad(mask, ((pad_h, pad_h), (pad_w, pad_w)),
                    mode="constant", constant_values=0)
    out = np.zeros((H, W), dtype=np.uint8)
    for dy in range(sH):
        for dx in range(sW):
            if se[dy, dx]:
                out |= padded[dy:dy+H, dx:dx+W]
    return out


def opening(mask, se):
    """Abertura = erode depois dilata."""
    return dilate(erode(mask, se), se)


def closing(mask, se):
    """Fechamento = dilata depois erode."""
    return erode(dilate(mask, se), se)


def fill_holes(mask):
    """Preenche buracos internos via flood fill do exterior (BFS iterativo)."""
    H, W = mask.shape
    padded = np.pad(mask, 1, mode="constant", constant_values=0)
    visited = np.zeros_like(padded, dtype=bool)

    q = deque()
    q.append((0, 0))
    visited[0, 0] = True
    while q:
        y, x = q.popleft()
        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            ny, nx = y + dy, x + dx
            if 0 <= ny < padded.shape[0] and 0 <= nx < padded.shape[1]:
                if not visited[ny, nx] and padded[ny, nx] == 0:
                    visited[ny, nx] = True
                    q.append((ny, nx))

    holes = (~visited) & (padded == 0)
    filled = padded | holes.astype(np.uint8)
    return filled[1:-1, 1:-1].astype(np.uint8)


# ── Demonstração das operações morfológicas ──────────────────────────────────
img = load_image("0619")
lab = to_lab(img)
lab[..., 2] = gaussian_lowpass_fft(lab[..., 2], sigma=25.0)
feat = warmth_feature(lab)
sp = slic_superpixels(lab, n_segments=300, compactness=12.0, n_iters=8)
coarse, _, _ = otsu_per_superpixel(sp, feat)

m1 = opening(coarse, disk_se(2))
m2 = fill_holes(m1)
m3 = closing(m2, disk_se(2))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].imshow(coarse, cmap="gray"); axes[0,0].set_title("Coarse (Otsu por superpixel)")
axes[0,1].imshow(m1, cmap="gray");     axes[0,1].set_title("Após abertura (disco r=2)")
axes[1,0].imshow(m2, cmap="gray");     axes[1,0].set_title("Após fill_holes")
axes[1,1].imshow(m3, cmap="gray");     axes[1,1].set_title("Após fechamento (disco r=2)")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "05_morfologia.png"))
plt.show()


## Bloco 7 — Componentes conectados, filtragem por área e contagem

### Componentes conectados (BFS, 4-conectividade)

Para cada pixel não rotulado de foreground, inicio um BFS que rotula todos os
pixels conectados a ele. Uso **4-conectividade** (cima/baixo/esq/dir) em vez de
8-conectividade porque em 8-conex dois grãos só "tocando" diagonalmente seriam
considerados um único componente.

### Filtro por área

Após CCs, removo:
- **min_area = 80 px**: pingos isolados (ruído residual que escapou do Otsu).
- **max_area = 25000 px**: a borda da placa de Petri, que vira um anel imenso
  e seria contado como grão. Não posso usar um max_area muito apertado porque
  clusters de grãos colados podem ser legitimamente grandes.

### Contagem com erosão

Pra contar grãos colados como entidades separadas: aplico uma **erosão extra**
no resultado final (raio 3) e conto componentes. A ponte fina entre dois grãos
desaparece com a erosão, mas o núcleo de cada grão sobrevive.

Esse truque tem limite — quando os grãos se sobrepõem profundamente (não só
se tocam), nem essa erosão separa. Discuto isso na análise.

### Métricas

- **IoU** = |A ∩ B| / |A ∪ B|
- **Dice** = 2·|A ∩ B| / (|A| + |B|)

Implemento as duas em uma linha cada — não precisa muito.


In [ ]:
def connected_components(mask):
    """Rotula CCs com 4-conectividade via BFS iterativo.

    Devolve (labels_array, n_components). Label 0 = fundo.
    """
    H, W = mask.shape
    labels = np.zeros((H, W), dtype=np.int32)
    next_label = 0
    for y in range(H):
        for x in range(W):
            if mask[y, x] and labels[y, x] == 0:
                next_label += 1
                q = deque()
                q.append((y, x))
                labels[y, x] = next_label
                while q:
                    cy, cx = q.popleft()
                    for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < H and 0 <= nx < W and mask[ny, nx] and labels[ny, nx] == 0:
                            labels[ny, nx] = next_label
                            q.append((ny, nx))
    return labels, next_label


def filter_components_by_area(mask, min_area, max_area=None):
    """Tira componentes fora do intervalo [min_area, max_area]."""
    labels, n = connected_components(mask)
    if n == 0:
        return mask.copy()
    areas = np.bincount(labels.ravel(), minlength=n + 1)
    keep = areas >= min_area
    if max_area is not None:
        keep &= areas <= max_area
    keep[0] = False  # fundo sempre fora
    return keep[labels].astype(np.uint8)


def count_grains_eroded(mask, erosion_radius=3, min_seed_area=10):
    """Conta grãos aplicando erosão antes da rotulagem — separa colados."""
    eroded = erode(mask, disk_se(erosion_radius))
    if min_seed_area > 0:
        eroded = filter_components_by_area(eroded, min_seed_area, None)
    _labels, n = connected_components(eroded)
    return n


def iou_score(pred, gt):
    """Intersection over Union."""
    p = pred.astype(bool); g = gt.astype(bool)
    inter = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    if union == 0:
        return 1.0 if inter == 0 else 0.0
    return float(inter) / float(union)


def dice_score(pred, gt):
    """Dice = 2·|A ∩ B| / (|A| + |B|)."""
    p = pred.astype(bool); g = gt.astype(bool)
    inter = np.logical_and(p, g).sum()
    s = p.sum() + g.sum()
    return 1.0 if s == 0 else 2.0 * float(inter) / float(s)


print("Funções de CC, contagem e métricas definidas.")


## Bloco 8 — Pipeline completo

Encapsulo todas as etapas em uma única função `run_pipeline(image_id)` que devolve
um dicionário com cada resultado intermediário (pra eu poder visualizar e depurar).

### Parâmetros finais

| Parâmetro | Valor | Onde |
|---|---|---|
| sigma do passa-baixa | 25 | FFT |
| n_segments | 300 | SLIC |
| compactness | 12 | SLIC |
| n_iters | 8 | SLIC |
| l_low, l_high | 20, 40 | gate de luminosidade |
| SE da abertura | disco r=2 | morfologia |
| min_area | 80 | filtro de componentes |
| max_area | 25000 | filtro de componentes |
| SE do fechamento | disco r=2 | morfologia |
| erosion_radius da contagem | 3 | contagem |


In [ ]:
def run_pipeline(image_id, verbose=False):
    """Executa o pipeline completo numa imagem e devolve um dict com tudo."""
    import time as _t

    img = load_image(image_id)
    H, W = img.shape[:2]

    # GT a partir das bndbox
    boxes = parse_annotation(image_id, W, H)
    gt = build_gt_mask(boxes, (H, W))

    # 1) Pré-processamento — LAB
    lab = to_lab(img)

    # 2) Filtragem na frequência — passa-baixa no b*
    b_smooth = gaussian_lowpass_fft(lab[..., 2], sigma=25.0)
    lab[..., 2] = b_smooth

    # 3) Construção da feature (b* * gate(L*))
    feat = warmth_feature(lab)

    # 4) SLIC
    if verbose: t = _t.time()
    sp = slic_superpixels(lab, n_segments=300, compactness=12.0, n_iters=8)
    if verbose: print(f"  SLIC: {_t.time()-t:.2f}s ({sp.max()+1} sp)")

    # 5) Otsu por superpixel
    coarse, t_otsu, _ = otsu_per_superpixel(sp, feat)

    # 6) Morfologia
    if verbose: t = _t.time()
    mask = opening(coarse, disk_se(2))
    mask = filter_components_by_area(mask, min_area=80, max_area=25000)
    mask = fill_holes(mask)
    mask = closing(mask, disk_se(2))
    if verbose: print(f"  Morfologia: {_t.time()-t:.2f}s")

    # 7) Contagem
    n_pred = count_grains_eroded(mask, erosion_radius=3, min_seed_area=10)
    n_gt = len(boxes)

    # 8) Métricas
    iou = iou_score(mask, gt)
    dice = dice_score(mask, gt)

    return dict(
        image_id=image_id, img=img, lab=lab, b_smooth=b_smooth, feature=feat,
        sp=sp, otsu_t=t_otsu, coarse=coarse, mask=mask,
        gt=gt, n_pred=n_pred, n_gt=n_gt, iou=iou, dice=dice
    )


# Roda no conjunto selecionado
results = []
for image_id in SELECTED_IMAGES:
    print(f"Processando {image_id}...")
    r = run_pipeline(image_id, verbose=True)
    print(f"  IoU={r['iou']:.3f}  Dice={r['dice']:.3f}  Contagem: pred={r['n_pred']} GT={r['n_gt']}")
    results.append(r)


## Bloco 8.1 — Teste manual e aleatório

Estas células deixam o notebook pronto para a apresentação: se o professor pedir
para rodar o pipeline em outra imagem, basta alterar `TEST_IMAGE_ID` ou ativar o
teste aleatório.

Por padrão, os testes extras ficam desligados para não aumentar o tempo de
execução quando o notebook for rodado inteiro.


In [ ]:
# Teste manual com qualquer imagem do dataset
# Para usar: troque RUN_TESTE_MANUAL para True e altere TEST_IMAGE_ID.

RUN_TESTE_MANUAL = False
TEST_IMAGE_ID = "0619"

if RUN_TESTE_MANUAL:
    print(f"Rodando teste manual na imagem {TEST_IMAGE_ID}...")
    resultado_teste = run_pipeline(TEST_IMAGE_ID, verbose=True)
    print(f"IoU={resultado_teste['iou']:.3f} | Dice={resultado_teste['dice']:.3f} | "
          f"Contagem: pred={resultado_teste['n_pred']} GT={resultado_teste['n_gt']}")
else:
    print("Teste manual desativado. Para usar, defina RUN_TESTE_MANUAL = True.")


In [ ]:
# Teste aleatório com qualquer imagem disponível no dataset
# Para usar: troque RUN_TESTE_ALEATORIO para True.

RUN_TESTE_ALEATORIO = False

if RUN_TESTE_ALEATORIO:
    all_image_ids = [p.stem for p in IMAGES_PATH.glob("*.jpg")]
    random_image_id = random.choice(all_image_ids)
    print("Imagem aleatória escolhida:", random_image_id)

    resultado_aleatorio = run_pipeline(random_image_id, verbose=True)
    print(f"IoU={resultado_aleatorio['iou']:.3f} | Dice={resultado_aleatorio['dice']:.3f} | "
          f"Contagem: pred={resultado_aleatorio['n_pred']} GT={resultado_aleatorio['n_gt']}")
else:
    print("Teste aleatório desativado. Para usar, defina RUN_TESTE_ALEATORIO = True.")


## Bloco 9 — Visualizações finais

Mostro pra cada imagem: original, feature, SLIC, máscara final e GT, com overlay
das fronteiras dos grãos detectados sobre a imagem original.


In [ ]:
for r in results:
    img = r["img"]
    sp = r["sp"]
    edges = np.zeros(sp.shape, dtype=bool)
    edges[:-1, :] |= sp[:-1, :] != sp[1:, :]
    edges[:, :-1] |= sp[:, :-1] != sp[:, 1:]

    # contorno do mask sobre o original
    mask = r["mask"]
    cy = np.zeros_like(mask); cx = np.zeros_like(mask)
    cy[:-1, :] = mask[:-1, :] != mask[1:, :]
    cx[:, :-1] = mask[:, :-1] != mask[:, 1:]
    contour = (cy | cx).astype(bool)

    overlay = img.copy()
    overlay[contour] = [255, 0, 0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    axes[0,0].imshow(img);                        axes[0,0].set_title(f"Original {r['image_id']}")
    axes[0,1].imshow(r["feature"], cmap="RdYlBu_r"); axes[0,1].set_title("feature = b*·gate(L*)")
    axes[0,2].imshow(sp % 47, cmap="tab20");      axes[0,2].set_title(f"SLIC ({sp.max()+1} sp)")
    axes[1,0].imshow(r["coarse"], cmap="gray");   axes[1,0].set_title("Coarse (Otsu)")
    axes[1,1].imshow(r["mask"], cmap="gray");     axes[1,1].set_title("Após morfologia")
    axes[1,2].imshow(overlay);                    axes[1,2].set_title("Overlay sobre original")
    for ax in axes.ravel(): ax.axis("off")
    plt.suptitle(f"{r['image_id']}  —  IoU={r['iou']:.3f}  Dice={r['dice']:.3f}  "
                 f"Contagem: pred={r['n_pred']} GT={r['n_gt']}",
                 fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"resultado_{r['image_id']}.png"))
    plt.show()


## Bloco 10 — Tabela de resultados e discussão

### Resumo quantitativo


In [ ]:
print(f"{'Imagem':>8} | {'IoU':>5} | {'Dice':>5} | {'Pred':>4} | {'GT':>3} | {'Erro contagem':>13}")
print("-" * 60)
ious = []; dices = []; errs = []
for r in results:
    err = abs(r["n_pred"] - r["n_gt"])
    ious.append(r["iou"]); dices.append(r["dice"]); errs.append(err)
    print(f"{r['image_id']:>8} | {r['iou']:>5.3f} | {r['dice']:>5.3f} | "
          f"{r['n_pred']:>4} | {r['n_gt']:>3} | {err:>13d}")
print("-" * 60)
print(f"{'Média':>8} | {np.mean(ious):>5.3f} | {np.mean(dices):>5.3f} | "
      f"{'':>4} | {'':>3} | {np.mean(errs):>13.1f}")


## Discussão crítica

### O que funcionou bem

- **Feature `b* × gate(L*)`**: foi o passo que mais aumentou o IoU. Antes do gate
  de luminosidade, imagens com vinheta escura (1113) tinham IoU ≈ 0.15. Com o gate,
  pulou pra ≈ 0.50. A insight chave foi perceber que o b* sozinho é insuficiente
  quando há regiões escuras na imagem.

- **SLIC + Otsu por superpixel**: a separação grão/fundo é praticamente perfeita
  visualmente. O histograma de médias por superpixel é bem bimodal, o que dá
  margem grande pra Otsu acertar o limiar.

- **Filtro de área**: simples e eficiente pra eliminar o anel da placa de Petri.

### O que não funcionou tão bem

- **Contagem em clusters densos**: em imagens com grãos fortemente sobrepostos
  (1141, 1286), a contagem subestima muito (10/31 e 12/47). A erosão antes do CC
  ajuda só em "toques" leves — quando os grãos se sobrepõem 30-40%, nada além de
  uma técnica mais sofisticada (watershed com markers de distância, por exemplo)
  separa eles.

- **GT aproximado por elipses**: como o dataset só tem bndbox, a elipse inscrita
  é uma aproximação grosseira. Em particular, ela:
  - Inflama o GT (a elipse cobre área que pode ser fundo).
  - Não captura grãos parcialmente sobrepostos corretamente.

  Isso significa que o IoU "real" do pipeline (se tivesse máscara verdadeira) seria
  provavelmente um pouco diferente — não dá pra dizer com certeza se pra cima ou
  pra baixo sem inspecionar manualmente.

- **Brotos/sprouts dos grãos germinados**: alguns grãos têm raízes brancas finas
  que o passa-baixa σ=25 borra e o Otsu não pega. Em imagens com muitos grãos
  germinados (1113, 1141), perde-se essa estrutura.

### Possíveis melhorias futuras

1. **Watershed por marcadores de distância**: implementar distance transform e
   usar local maxima como markers. Resolveria a contagem em clusters.
2. **Detecção da borda da placa via Hough Circles**: mais robusto que filtro de
   área pra eliminar a placa.
3. **Resolução maior (1280×720)**: melhoraria a fidelidade das bordas dos grãos,
   à custa de tempo de processamento.
4. **Combinar IoU em múltiplos canais**: usar b* e (-a*) juntos pra reforçar a
   tonalidade quente dos grãos.

### Impacto de cada etapa

| Etapa removida | Efeito no IoU médio |
|---|---|
| Sem filtro de frequência | IoU cai ~5% — SLIC fica ruidoso |
| Sem gate(L*) | IoU cai ~15% em imagens com vinheta |
| Sem fill_holes | IoU cai ~3% (poucos buracos) |
| Sem filtro de área (sem max) | IoU cai drasticamente (>30%) — borda da placa contamina tudo |
